Se agrega la raíz del proyecto al `sys.path` para poder hacer `from src import ...`
tanto si el notebook está en la raíz como si está en una carpeta `notebooks/`.

In [ ]:
import sys
from pathlib import Path

raiz = Path.cwd()
if not (raiz / "src").is_dir():
    raiz = raiz.parent
sys.path.append(str(raiz))

from src import config, data_loader, profiling, tendencias

print("Raíz del proyecto:", raiz)
print("Carpeta de datos :", config.DIR_DATOS)
print("Carpeta de salida:", config.DIR_SALIDA)

In [ ]:
ANIO = "2024"
CATEGORIA = config.CAT_MERCADO_LABORAL

## 2. Detección de archivos y verificación de trimestres

`buscar_archivos` recorre `data/` y devuelve los archivos reconocidos.
`trimestres_faltantes` comprueba que haya hogar individuo en cada trimestre.

In [ ]:
archivos_anio = data_loader.buscar_archivos(categoria=CATEGORIA, anios=[ANIO])

for a in archivos_anio:
    print(a["anio"], a["trimestre"], a["tipo"], a["ruta"].name)

In [ ]:
faltantes = data_loader.trimestres_faltantes(archivos_anio, CATEGORIA, ANIO)

if faltantes:
    print("Faltan archivos en los trimestres:", faltantes)
else:
    print("El año", ANIO, "tiene los 4 trimestres completos (hogar e individuo).")

In [ ]:
archivos_ind = data_loader.filtrar_archivos(archivos_anio, tipo=config.TIPO_INDIVIDUO)
print("Archivos de individuos a procesar:", len(archivos_ind))

## 3. Exploración de la estructura

### 3.1 Columnas
Se leen solo los encabezados y se compara que sean iguales entre trimestres.

In [ ]:
columnas_por_trimestre = {}
for a in archivos_ind:
    columnas_por_trimestre[a["trimestre"]] = data_loader.leer_columnas(a["ruta"])

for trimestre in columnas_por_trimestre:
    print(trimestre, "->", len(columnas_por_trimestre[trimestre]), "columnas")

primer = archivos_ind[0]["trimestre"]
iguales = True
for trimestre in columnas_por_trimestre:
    if sorted(columnas_por_trimestre[trimestre]) != sorted(columnas_por_trimestre[primer]):
        iguales = False
        print("Difieren las columnas de", trimestre, "respecto de", primer)
print("Columnas idénticas entre trimestres:", iguales)

In [ ]:
columnas = columnas_por_trimestre[primer]

# Variables que usa el análisis (A.1, A.2, A.3 y B.1)
a_revisar = list(config.VARS_INDIVIDUOS)
a_revisar += [config.COL_SEXO, config.COL_INGRESO, config.COL_INGRESO_OTRAS,
              config.COL_INGRESO_TOTAL, "CODUSU", "NRO_HOGAR", "COMPONENTE"]

for nombre in a_revisar:
    estado = "OK" if nombre in columnas else "NO EXISTE"
    print(f"{nombre:<12} {estado}")

### 3.2 Valores de las variables de estado

`profiling.frecuencias` cuenta cuántas veces aparece cada valor. Sirve para decidir
qué códigos son "no corresponde" y cuáles son faltantes reales (ver `config.py`).


In [ ]:
filas = data_loader.leer_filas(archivos_ind[0]["ruta"])
print("Trimestre:", archivos_ind[0]["trimestre"], "- filas:", len(filas))

for columna in (config.COL_ESTADO, config.COL_CAT_OCUP, config.COL_CAT_INAC):
    print("\n" + columna)
    conteo = profiling.frecuencias(filas, columna)
    for valor in sorted(conteo):
        print("  ", valor, "->", conteo[valor])

In [ ]:
# ESTADO x CAT_OCUP: sirve para decidir qué es "no corresponde" y qué es "faltante"
cruce = {}
for fila in filas:
    clave = (fila.get(config.COL_ESTADO), fila.get(config.COL_CAT_OCUP))
    cruce[clave] = cruce.get(clave, 0) + 1

for clave in sorted(cruce, key=str):
    print("ESTADO =", clave[0], "| CAT_OCUP =", repr(clave[1]), "->", cruce[clave])

del filas
# se libera la memoria antes del procesamiento completo

## 4. Procesamiento (una sola lectura por archivo)

Cada archivo se lee **una vez** y, en esa misma pasada, se calcula todo lo de la
Parte A:
- `profiling.contar_valores`: conteos de A.1 (válidos, faltantes reales y no corresponde).
- `profiling.evaluar_reglas`: inconsistencias de A.2 y A.3 (con las reglas de la Modificación 5).
- `profiling.frecuencias` de ESTADO: cantidad de desocupados (para la línea informativa de A.3.2).

Se guardan **solo los conteos**, no las filas: guardar los 4 trimestres completos en
memoria no hace falta y leer cada archivo una vez por requerimiento multiplicaría el
tiempo. Como se guardan conteos, los porcentajes se pueden recalcular con cualquier
umbral y los trimestres se pueden sumar para obtener el año.

In [ ]:
REGLAS = profiling.REGLAS_A2 + profiling.REGLAS_A3

resultados_a1 = []
resultados_consistencia = []
desocupados_por_trimestre = {}
for a in archivos_ind:
    filas = data_loader.leer_filas(a["ruta"])
    resultados_a1.append({
        "anio": a["anio"],
        "trimestre": a["trimestre"],
        "conteos": profiling.contar_valores(filas, config.VARS_INDIVIDUOS),
    })
    resultados_consistencia.append({
        "anio": a["anio"],
        "trimestre": a["trimestre"],
        "hallazgos": profiling.evaluar_reglas(filas, REGLAS),
    })
    estados = profiling.frecuencias(filas, config.COL_ESTADO)
    desocupados_por_trimestre[a["trimestre"]] = estados.get(config.ESTADO_DESOCUPADO, 0)
    print(a["anio"], a["trimestre"], "->", len(filas), "filas procesadas")
del filas

## 5. Requerimiento A.1 - Completitud

### 5.1 Por trimestre

In [ ]:
def mostrar_completitud(titulo, res):
    """Imprime la tabla de completitud de un período."""
    print("=" * 64)
    print(titulo, "- filas:", res["total_filas"], "- umbral:", res["umbral_pct"], "%")
    print(f"{'columna':<14}{'validos':>9}{'faltantes':>11}{'no_corr.':>10}{'% falt.':>9}")
    for columna, d in res["columnas"].items():
        print(f"{columna:<14}{d['validos']:>9}{d['faltantes_reales']:>11}"
              f"{d['no_corresponde']:>10}{d['pct_faltantes']:>9}")
    print("Superan el umbral:", res["sobre_umbral"])
    print("Mayor % de faltantes:", res["mayor_faltante"], f"({res['pct_mayor_faltante']} %)")
    if res["inexistentes"]:
        print("ATENCIÓN, columnas inexistentes:", res["inexistentes"])


for r in resultados_a1:
    res = profiling.completitud_con_umbral(r["conteos"])
    mostrar_completitud(r["anio"] + " " + r["trimestre"], res)

### 5.2 Anual

El año se calcula **sumando los conteos** de los 4 trimestres (`sumar_conteos`) y
recién después se calcula el porcentaje. Promediar los porcentajes trimestrales daría
otro resultado, porque cada trimestre tiene distinta cantidad de filas.

In [ ]:
conteos_anuales = profiling.sumar_conteos([r["conteos"] for r in resultados_a1])
mostrar_completitud("Año " + ANIO, profiling.completitud_con_umbral(conteos_anuales))

### 5.3 Varios umbrales (Modificación 4)

Los umbrales se configuran en `config.UMBRALES_FALTANTES_PCT`. Se genera un archivo por
umbral con el nombre `config.ARCHIVO_CALIDAD` (`calidad_datos_umbral_{umbral}.json`).
Cada informe se guarda en la sección `completitud_individuos`, así no se pisa la de
hogares que escribe el notebook 2. Además, el reporte `calidad_datos.json` que pide el
enunciado se genera con el umbral por defecto (`config.UMBRAL_FALTANTES_PCT`).

In [ ]:
metadatos_a1 = {
    "categoria": CATEGORIA,
    "tipo_archivo": config.TIPO_INDIVIDUO,
    "anios": [ANIO],
    "variables": config.VARS_INDIVIDUOS,
}

for umbral in config.UMBRALES_FALTANTES_PCT:
    informe = profiling.armar_informe_completitud(resultados_a1, umbral, metadatos_a1)
    ruta = config.DIR_SALIDA / config.ARCHIVO_CALIDAD.format(umbral=umbral)
    data_loader.guardar_reporte("completitud_individuos", informe, ruta)
    print(f"Umbral {umbral:>2} % -> {ruta.name} | superan en {ANIO}:",
          informe["anual"][ANIO]["sobre_umbral"])

informe = profiling.armar_informe_completitud(
    resultados_a1, config.UMBRAL_FALTANTES_PCT, metadatos_a1)
data_loader.guardar_reporte("completitud_individuos", informe)
print("Guardado también en", config.RUTA_REPORTE.name)

## 6. Requerimiento A.2 - Consistencia edad-trabajo

In [ ]:
def mostrar_hallazgos(titulo, hallazgos, reglas, con_ejemplos=False):
    """Imprime las reglas pedidas de un período, ordenadas por gravedad."""
    nombres = [regla["nombre"] for regla in reglas]
    print("=" * 78)
    print(titulo, "- filas:", hallazgos["total_filas"])
    for nombre in profiling.ordenar_por_gravedad(hallazgos["reglas"]):
        if nombre not in nombres:
            continue
        d = hallazgos["reglas"][nombre]
        print(f"[{d['etiqueta']:<11}] {d['requerimiento']:<9}{d['cantidad']:>7}  {d['descripcion']}")
        for valor, cantidad in d.get("desagregado", {}).items():
            print(f"{'':>30}{config.NOMBRES_SEXO.get(valor, valor)}: {cantidad}")
        if con_ejemplos:
            for ejemplo in d["ejemplos"]:
                print("        ", ejemplo)


for r in resultados_consistencia:
    mostrar_hallazgos(r["anio"] + " " + r["trimestre"], r["hallazgos"], profiling.REGLAS_A2)

In [ ]:
### 6.1 Anual, con ejemplos

In [ ]:
hallazgos_anuales = profiling.sumar_hallazgos(
    [r["hallazgos"] for r in resultados_consistencia])
mostrar_hallazgos("Año " + ANIO, hallazgos_anuales, profiling.REGLAS_A2, con_ejemplos=True)

## 7. Requerimiento A.3 - Consistencia entre variables relacionadas

In [ ]:
for r in resultados_consistencia:
    mostrar_hallazgos(r["anio"] + " " + r["trimestre"], r["hallazgos"], profiling.REGLAS_A3)

In [ ]:
mostrar_hallazgos("Año " + ANIO, hallazgos_anuales, profiling.REGLAS_A3, con_ejemplos=True)

### 7.1 Ingresos de los desocupados

A.3.2 usa P21. Como complemento informativo decidimos mostrar cuántos desocupados tienen
ingreso de otras ocupaciones (TOT_P12) o ingreso total individual (P47T).

In [ ]:
for r in resultados_consistencia:
    trimestre = r["trimestre"]
    reglas = r["hallazgos"]["reglas"]
    print(f"Desocupados {trimestre}: {desocupados_por_trimestre[trimestre]}"
          f" | con P21 > 0: {reglas['desocupado_con_ingreso']['cantidad']}"
          f" | con TOT_P12 informado: {reglas['desocupado_con_otras_ocupaciones']['cantidad']}"
          f" | con P47T > 0: {reglas['desocupado_con_ingreso_total']['cantidad']}")

### 7.2 Reporte `calidad_datos.json`

Se guarda A.2 y A.3 (por trimestre y anual, ordenado por gravedad y con ejemplos) en la
sección `consistencia_individuos` del reporte.

In [ ]:
metadatos_consistencia = {
    "categoria": CATEGORIA,
    "tipo_archivo": config.TIPO_INDIVIDUO,
    "anios": [ANIO],
    "ingreso_usado": config.COL_INGRESO,
}
informe = profiling.armar_informe_consistencia(resultados_consistencia, metadatos_consistencia)
data_loader.guardar_reporte("consistencia_individuos", informe)
print("Guardado en", config.RUTA_REPORTE)

## 8. Informe por roles (Modificación 5.4)

Los roles se definen en `config.ROLES` (rol -> columnas que le interesan). Para cada
columna se juntan las reglas que la usan y que tienen una etiqueta de inconsistencia
(`config.ETIQUETAS_INCONSISTENCIA`: CRÍTICO, ERROR y VERIFICAR). Para agregar un rol
alcanza con agregar una entrada en `config.ROLES`.

ITF no tiene reglas de consistencia en el archivo de individuos (solo se analiza su
completitud en hogares), por eso aparece con 0 casos.

In [ ]:
roles = profiling.informe_por_roles(hallazgos_anuales)

for rol, datos in roles.items():
    print("=" * 60)
    print("Rol:", rol)
    for columna, d in datos["columnas"].items():
        print(f"  {columna:<10}{d['casos']:>7} casos  ->", list(d["reglas"]))
    print("  Columnas con inconsistencias:", datos["tiene_inconsistencias"])

ruta_roles = config.DIR_SALIDA / config.ARCHIVO_ROLES
data_loader.guardar_reporte("individuos_" + ANIO, roles, ruta_roles)
print("\nGuardado en", ruta_roles)

## 9. Requerimiento B.1 - Tendencias por aglomerado

In [ ]:
todos = data_loader.buscar_archivos(categoria=CATEGORIA)
individuos = data_loader.filtrar_archivos(todos, tipo=config.TIPO_INDIVIDUO)
disponibles = sorted(set(a["anio"] for a in individuos))
print("Años disponibles:", disponibles)
print("Aglomerados:")
for codigo, nombre in config.AGLOMERADOS.items():
    print(f"  {codigo:>3} {nombre}")

In [ ]:
texto_anios = input("Rango de años (ej. 2024 o 2023-2025; Enter = todos): ")
anios_b1, no_disponibles = tendencias.interpretar_anios(texto_anios, disponibles)
if no_disponibles:
    print("Sin archivos para:", no_disponibles)

texto_aglos = input("Aglomerados separados por coma (ej. 32, 33, 13; Enter = todos): ")
aglomerados_b1, invalidos = tendencias.interpretar_aglomerados(texto_aglos)
if invalidos:
    print("Códigos de aglomerado no reconocidos:", invalidos)

print("Años:", anios_b1)
print("Aglomerados:", [tendencias.nombre_aglomerado(c) for c in aglomerados_b1])

In [ ]:
archivos_b1 = data_loader.filtrar_archivos(individuos, anios=anios_b1)
for anio in anios_b1:
    faltan = data_loader.trimestres_faltantes(todos, CATEGORIA, anio)
    if faltan:
        print("ATENCIÓN:", anio, "no tiene completos los trimestres", faltan)

resultados_b1 = []
for a in archivos_b1:
    conteos = tendencias.contar_tendencias(data_loader.iterar_filas(a["ruta"]), aglomerados_b1)
    resultados_b1.append({"anio": a["anio"], "trimestre": a["trimestre"], "conteos": conteos})
    print(a["anio"], a["trimestre"], "procesado | sin PONDERA:", conteos["sin_pondera"])

In [ ]:
def formato(valor):
    """Muestra un porcentaje o '-' si no se pudo calcular."""
    return "-" if valor is None else f"{valor:.2f}"


def mostrar_linea(nombre, t):
    """Imprime la tasa de desempleo y la distribución de un grupo."""
    dist = t["distribucion_pct"]
    print(f"    {nombre:<18} desempleo {formato(t['tasa_desempleo']):>6} %"
          f" | ocup {formato(dist['ocupados']):>6} % desoc {formato(dist['desocupados']):>6} %"
          f" inac {formato(dist['inactivos']):>6} % | población {t['poblacion']:>10,}"
          f" (muestra {t['casos_muestra']})")


def mostrar_tendencias(aglomerados):
    """Imprime, por aglomerado y período, el total y las desagregaciones."""
    for codigo, datos in aglomerados.items():
        print("=" * 100)
        print(f"{datos['nombre']} (aglomerado {codigo})")
        for periodo, bloque in datos["periodos"].items():
            print(f"  {periodo}")
            mostrar_linea("Total", bloque["total"])
            for nombre, t in bloque["por_sexo"].items():
                mostrar_linea(nombre, t)
            for nombre, t in bloque["por_rango_etario"].items():
                mostrar_linea(nombre, t)


metadatos_b1 = {
    "categoria": CATEGORIA,
    "anios": anios_b1,
    "aglomerados": [tendencias.nombre_aglomerado(c) for c in aglomerados_b1],
    "ponderacion": config.COL_PONDERA,
    "rangos_etarios": [r[0] for r in config.RANGOS_ETARIOS],
}
informe_trimestral = tendencias.armar_informe_trimestral(resultados_b1, metadatos_b1)
mostrar_tendencias(informe_trimestral["aglomerados"])

### 9.1 Informe semestral y anual (Modificación 6.5)

Cada semestre (`config.SEMESTRES`) y cada año se calculan **sumando las personas
ponderadas** de sus trimestres y recién después se calculan las tasas. Si falta algún
trimestre, el período se informa igual y queda marcado con `completo = False`.

In [ ]:
informe_periodos = tendencias.armar_informe_periodos(resultados_b1, metadatos_b1)

print("SEMESTRAL")
mostrar_tendencias(informe_periodos["semestral"])
print("\nANUAL")
mostrar_tendencias(informe_periodos["anual"])

ruta_trim = config.DIR_SALIDA / config.ARCHIVO_TENDENCIAS_TRIMESTRAL
ruta_per = config.DIR_SALIDA / config.ARCHIVO_TENDENCIAS_PERIODOS
data_loader.guardar_reporte("tendencias_por_aglomerado", informe_trimestral, ruta_trim)
data_loader.guardar_reporte("tendencias_por_aglomerado", informe_periodos, ruta_per)
print("\nGuardado en", ruta_trim.name, "y", ruta_per.name)